# Setup

In [1]:
# IMPORTS
import numpy as np
import pandas as pd

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.graph_objects as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt

In [2]:
import gdown
# gdown.download("https://drive.google.com/uc?id=1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5", output="/content/",)

# 0) Dataset for Modeling: Final Preparations

## 0.1) Importing data from Drive & defining variable sets

In [3]:
df_full = pd.read_parquet("/content/stocks_df_combined_2026_09_18.parquet.brotli",)

In [4]:
GROWTH = [g for g in df_full.keys() if (g.find('growth_')==0)&(g.find('future')<0)]
OHLCV  = ['Open','High','Low','Close','Adj Close_x','Volume']
CATEGORICAL = ['Month', 'Weekday', 'Ticker', 'ticker_type']

TO_PREDICT = [g for g in df_full.keys() if (g.find('future')>=0)]
TO_DROP = ['Year','Date','index_x', 'index_y', 'index', 'Quarter','Adj Close_y'] + CATEGORICAL + OHLCV

In [5]:
df_full['ln_volume'] = df_full.Volume.replace(0, np.nan).fillna(1e-9).apply(lambda x: np.log(x))

In [6]:
TECHNICAL_INDICATORS = ['adx', 'adxr', 'apo', 'aroon_1','aroon_2', 'aroonosc',
 'bop', 'cci', 'cmo','dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext',
 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix',
 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo',
 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk',
 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr',
 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase',
 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine',
 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice']

In [7]:
TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find('cdl')>=0]

In [8]:
CUSTOM_NUMERICAL = ['SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative','volatility', 'ln_volume']
MACRO = ['gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS', 'DGS1', 'DGS5', 'DGS10']

NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO

In [9]:
df = df_full[df_full.Date>='2000-01-01']

## 0.2) [Code snippet 1] Generating dummies

In [10]:
df['Weekday'] = df['Weekday'].astype(str)
df['Month'] = df['Month'].dt.month.astype(str)

df['Week'] = (df.Date.dt.day - 1) // 7 + 1
df['Month_Name'] = pd.to_datetime(df['Month'], format='%m').dt.month_name()
df['month_wom'] = df['Month_Name'] + '_w' + df['Week'].astype(int).astype(str)

CATEGORICAL = CATEGORICAL + ['month_wom']

In [11]:
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')

In [12]:
DUMMIES = dummy_variables.keys().to_list()

df_with_dummies = pd.concat([df, dummy_variables], axis=1)

## 0.3) [Code Snippet 2] Correlation analysis

In [13]:
corr_is_positive_growth_30d_future = df_with_dummies[NUMERICAL+DUMMIES+TO_PREDICT].corr()['is_positive_growth_30d_future']

In [14]:
corr_is_positive_growth_30d_future_df = pd.DataFrame(corr_is_positive_growth_30d_future)

In [15]:
corr_is_positive_growth_30d_future_df = \
    corr_is_positive_growth_30d_future_df[
    corr_is_positive_growth_30d_future_df.index.astype(str).str.contains('month_wom', case=False, na=False)]\
    .sort_values(by='is_positive_growth_30d_future')

corr_is_positive_growth_30d_future_df['abs_corr'] = \
    abs(corr_is_positive_growth_30d_future_df.is_positive_growth_30d_future)

corr_is_positive_growth_30d_future_df.sort_values(by='abs_corr', ascending=False).round(3).head(1)

,is_positive_growth_30d_future,abs_corr
month_wom_October_w4,0.025,0.025


In [16]:
corr_growth_future_30d = df_with_dummies[NUMERICAL+DUMMIES+TO_PREDICT].corr()['growth_future_30d']

In [17]:
corr_growth_future_30d_df = pd.DataFrame(corr_growth_future_30d)

## 0.4) [Code snippet 3] Temporal split of ~25 years of data (by date)

In [18]:
def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)
    split_labels = []
    for date in df['Date']:
        if date <= train_end:
            split_labels.append('train')
        elif date <= val_end:
            split_labels.append('validation')
        else:
            split_labels.append('test')

    df['split'] = split_labels
    return df

In [19]:
min_date_df = df_with_dummies.Date.min()
max_date_df = df_with_dummies.Date.max()

df_with_dummies = temporal_split(df_with_dummies, min_date = min_date_df, max_date = max_date_df)

In [20]:
df_with_dummies['split'].value_counts()/len(df_with_dummies)

split
train         0.676458
test          0.163936
validation    0.159606
Name: count, dtype: float64

In [21]:
new_df = df_with_dummies.copy()

# 1) Modeling: "rule of thumb" or hand-predictions

## 1.1) Review all the inputs again

In [22]:
new_df = df_with_dummies.copy()

## 1.2) [Code Snippet 3] Manual "hand rule" predictions

In [23]:
new_df['pred0_manual_cci'] = (new_df.cci>200).astype(int)
new_df['pred1_manual_prev_g1'] = (new_df.growth_30d>1).astype(int)
new_df['pred2_manual_prev_g1_and_snp'] = ((new_df['growth_30d'] > 1) & (new_df['growth_snp500_30d'] > 1)).astype(int)
new_df['pred3_manual_dgs10_5'] = ((new_df.DGS10 <= 4.5) & (new_df.DGS5 <= 4)).astype(int)
new_df['pred4_manual_dgs10_fedfunds'] = ((new_df.DGS10 > 4) & (new_df.FEDFUNDS <= 4.795)).astype(int)

In [24]:
PREDICTIONS = [k for k in new_df.keys() if k.startswith('pred')]
IS_CORRECT =  [k for k in new_df.keys() if k.startswith('is_correct_')]

for i,column in enumerate(IS_CORRECT):
  prediction_column = PREDICTIONS[i]
  is_correct_column = column
  filter = (new_df.split=='test') & (new_df[prediction_column]==1)
  print((new_df[filter][is_correct_column].value_counts()/len(new_df[filter])).round(3))
  print('---------')

## 1.3) [Code Snippet 4 - Advanced] Statistical prediction : ARIMA models with 3 parameters (p,q,r)

## 1.4) [Code Snippet 5] Binary Decision Tree

In [25]:
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

### 1.4.1) Define dataframes AND perform data cleaning

In [26]:
features_list = NUMERICAL+DUMMIES
to_predict = 'is_positive_growth_30d_future'

train_df = new_df[new_df.split.isin(['train','validation'])].copy(deep=True)
test_df = new_df[new_df.split.isin(['test'])].copy(deep=True)

X_train = train_df[features_list+[to_predict,'Date','Ticker']]
X_test = test_df[features_list+[to_predict,'Date','Ticker']]

print(f'length: X_train {X_train.shape}, X_test {X_test.shape}')

length: X_train (169129, 302), X_test (33163, 302)


In [27]:
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_train.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)

print(f'length: X_train_imputed {X_train.shape},  X_test_imputed {X_test.shape}')

length: X_train_imputed (169129, 302),  X_test_imputed (33163, 302)


In [28]:
X_train_imputed = X_train
X_test_imputed = X_test

In [29]:
y_train = X_train_imputed[to_predict]
y_test = X_test_imputed[to_predict]

del X_train_imputed[to_predict]
del X_test_imputed[to_predict]

### 1.4.2 Estimation of a Decision Tree model

In [30]:
def fit_decision_tree(X, y, max_depth=10):
  clf = DecisionTreeClassifier(max_depth=max_depth, random_state=42) 
  clf.fit(X, y)
  return clf, X.columns

In [31]:
%%time
clf_20, train_columns = fit_decision_tree(X=X_train_imputed.drop(['Date','Ticker'],axis=1), y=y_train, max_depth=20)

CPU times: total: 33.1 s
Wall time: 34.8 s


In [32]:
%%time
clf_10, train_columns = fit_decision_tree(X=X_train_imputed.drop(['Date','Ticker'],axis=1), y=y_train, max_depth=10)

CPU times: total: 20.9 s
Wall time: 22 s


### 1.4.3 Inference for a Decision Tree

In [33]:
def predict_decision_tree(clf:DecisionTreeClassifier, df_X:pd.DataFrame, y_true: pd.Series):
  y_pred = clf.predict(df_X)

  max_depth = clf.tree_.max_depth
  print("Maximum depth of the decision tree:", max_depth)

  accuracy = accuracy_score(y_test, y_pred)
  precision = precision_score(y_test, y_pred)
  print(f'Accuracy ={accuracy}, precision = {precision}')

  result_df = pd.concat([df_X, y_true, pd.Series(y_pred, index=df_X.index, name='pred_')], axis=1)

  return result_df

In [34]:
pred20 = predict_decision_tree(clf_20, X_test_imputed.drop(['Date','Ticker'],axis=1), y_test)

Maximum depth of the decision tree: 20
Accuracy =0.4856918855350843, precision = 0.5910074773711137


In [35]:
pred10 = predict_decision_tree(clf_10, X_test_imputed.drop(['Date','Ticker'],axis=1), y_test)

Maximum depth of the decision tree: 10
Accuracy =0.4773693574163978, precision = 0.5912146499768196


In [50]:
global_NUMERICAL = globals().get('NUMERICAL', [])
global_DUMMIES = globals().get('DUMMIES', [])
global_TO_PREDICT = globals().get('TO_PREDICT', [])

features_list_from_globals = global_NUMERICAL + global_DUMMIES
to_predict_col = global_TO_PREDICT[1] if len(global_TO_PREDICT) > 1 else 'is_positive_growth_30d_future'

if 'new_df' not in globals() or globals()['new_df'].empty:
    raise RuntimeError("new_df is not defined or is empty. Please run all preceding data loading and preprocessing cells.")

test_df = new_df[new_df.split.isin(['test'])].copy(deep=True)

valid_features = [col for col in features_list_from_globals if col in test_df.columns]

X_test_columns_to_select = valid_features + ['Date', 'Ticker']
if to_predict_col in test_df.columns and to_predict_col not in X_test_columns_to_select:
    X_test_columns_to_select.append(to_predict_col)

X_test = test_df[X_test_columns_to_select].copy()

pd.options.mode.chained_assignment = None
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(0, inplace=True)
pd.options.mode.chained_assignment = 'warn'

X_test_imputed = X_test.copy()

if to_predict_col in X_test_imputed.columns:
    del X_test_imputed[to_predict_col]

print(f'X_test_imputed has been recreated with shape: {X_test_imputed.shape}')

X_test_imputed has been recreated with shape: (1491265, 301)


In [37]:
date_ticker_info = X_test_imputed[['Date', 'Ticker']]

pred10_with_date_ticker = pred10.join(date_ticker_info)
pred20_with_date_ticker = pred20.join(date_ticker_info)

pred10_df = pred10_with_date_ticker[['Date', 'Ticker']].copy()
pred10_df['pred_tree_clf10'] = pred10_with_date_ticker['pred_'].values

pred20_df = pred20_with_date_ticker[['Date', 'Ticker']].copy()
pred20_df['pred_tree_clf20'] = pred20_with_date_ticker['pred_'].values

new_df = new_df.merge(
    pred10_df[['Date', 'Ticker', 'pred_tree_clf10']],
    on=['Date', 'Ticker'],
    how='left'
)

new_df = new_df.merge(
    pred20_df[['Date', 'Ticker', 'pred_tree_clf20']],
    on=['Date', 'Ticker'],
    how='left'
)

### 1.4.4 Features Importance and Tree Visualisation of top levels (for clf10)

### 1.4.5 Merge with the original df for predictions (only when predicted on test dataset)